In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
DATA_DIR = Path("../data/processed")

train_df = pd.read_pickle(DATA_DIR / "train.pkl")
valid_df = pd.read_pickle(DATA_DIR / "valid.pkl")
test_df  = pd.read_pickle(DATA_DIR / "test.pkl")

target_col = "Money Laundering Risk Score"

y_train = train_df[target_col].astype(int)
y_valid = valid_df[target_col].astype(int)
y_test  = test_df[target_col].astype(int)

In [9]:
categorical_features = [
    "Country",
    "Transaction Type",
    "Industry",
    "Destination Country",
    "Tax Haven Country",
    "Financial Institution_grouped",
]

numeric_features = [
    "Shell Companies Involved",
    "Amount (USD)",
    "transaction_year",
    "transaction_month",
    "transaction_dayofweek",
    "transaction_hour",
    "is_illegal",
    "is_reported",
]

In [10]:
X_train_raw = train_df[numeric_features + categorical_features].copy()
X_valid_raw = valid_df[numeric_features + categorical_features].copy()
X_test_raw  = test_df[numeric_features + categorical_features].copy()

In [11]:
encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    X_train_raw[col] = le.fit_transform(X_train_raw[col].astype(str))
    X_valid_raw[col] = le.transform(X_valid_raw[col].astype(str))
    X_test_raw[col]  = le.transform(X_test_raw[col].astype(str))
    encoders[col] = le

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
RANDOM_STATE = 42
logreg_simple = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

logreg_simple.fit(X_train_raw, y_train)

valid_pred = logreg_simple.predict(X_valid_raw)
test_pred  = logreg_simple.predict(X_test_raw)

valid_acc = accuracy_score(y_valid, valid_pred)
test_acc  = accuracy_score(y_test,  test_pred)

valid_f1 = f1_score(y_valid, valid_pred, average="weighted")
test_f1  = f1_score(y_test,  test_pred,  average="weighted")

valid_acc, valid_f1, test_acc, test_f1

(0.10266666666666667,
 0.07419634496211962,
 0.11333333333333333,
 0.08175977975488229)

In [13]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_simple, MODEL_DIR / "baseline.pkl")

['..\\models\\baseline.pkl']